In [3]:
import feedparser
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

clean_path = "../data/clean/"
raw_path = "../data/raw/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
pdf_path = "../PDF/"

pd.set_option('display.max_colwidth', 255)
pd.set_option('display.max_rows',None)
url = "https://feeds.feedburner.com/Setorth-form45-en"

today = date.today()
year = 2022
mmdd_str = today.strftime('%m%d')
mmdd_str

'0124'

In [4]:
#today = date(2023, 1, 20)
mmdd_str = today.strftime('%m%d')
mmdd_str

'0124'

In [5]:
rss_source = feedparser.parse(url)
f45_number = len(rss_source.entries)
print("Number of F45: ", f45_number)

Number of F45:  4


In [6]:
f45_items = []

for x in range(f45_number):
    f45_content = rss_source.entries[x]
    f45_item = {}
    
    print("\n----------------------------------\n")
    
    print("F45: " + str(x))
    title_ary = f45_content.title.partition(' ')
    f45_item['name'] = title_ary[0].strip() 
    print("Title: ", f45_item['name'])  
    f45_item['year'] = year
    print("Year: ", f45_item['year'])      
    qtr_ary = title_ary[2].partition(' (F45)')
    f45_item['quarter'] = qtr_ary[0][-1]    
    print("Quarter: ", f45_item['quarter'])    
    f45_item['link'] = f45_content.link
    print("Link: ", f45_item['link'])
    f45_item['published'] = f45_content.published
    print("Published: ", f45_item['published'])  
    f45_items.append(f45_item)


----------------------------------

F45: 0
Title:  COTTO
Year:  2022
Quarter:  2
Link:  https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174321600
Published:  Tue, 24 Jan 2023 13:38:21 +0700

----------------------------------

F45: 1
Title:  SCGP
Year:  2022
Quarter:  S
Link:  https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174320110
Published:  Tue, 24 Jan 2023 12:37:52 +0700

----------------------------------

F45: 2
Title:  MASTER
Year:  2022
Quarter:  3
Link:  https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174317110
Published:  Tue, 24 Jan 2023 08:32:02 +0700

----------------------------------

F45: 3
Title:  MASTER
Year:  2022
Quarter:  y
Link:  https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174317090
Published:  Tue, 24 Jan 2023 08:31:46 +0700


In [7]:
df = pd.DataFrame(f45_items)
df[['name','year','quarter','published']]

,name,year,quarter,published
0,COTTO,2022,2,"Tue, 24 Jan 2023 13:38:21 +0700"
1,SCGP,2022,S,"Tue, 24 Jan 2023 12:37:52 +0700"
2,MASTER,2022,3,"Tue, 24 Jan 2023 08:32:02 +0700"
3,MASTER,2022,y,"Tue, 24 Jan 2023 08:31:46 +0700"


In [6]:
df.groupby(['year','quarter']).count()

,,name,link,published
year,quarter,,,
2022,y,6,6,6


In [8]:
df.loc[(df.quarter == 'S') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 2           1     1          1
     3           1     1          1
     y           1     1          1
     4           1     1          1

In [11]:
df.loc[(df.quarter == 'y') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 3           1     1          1
     4           3     3          3

In [ ]:
df.loc[(df.name == 'MC') ,['year','quarter']] = ['2023','1']
df.groupby(['year','quarter']).count()

### After change all illogical quarters

In [12]:
df.quarter = df.quarter.astype(int)
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 3           1     1          1
     4           3     3          3

In [13]:
### First equals to latest published pdf file
df = df.drop_duplicates(subset='name', keep='first')
df.shape

(3, 5)

In [14]:
file_name = 'F45-RAW-' + mmdd_str + '.csv'
raw_file = raw_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

df[['name','year','quarter','published']].to_csv(output_file, header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(box_file,    header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(one_file,    header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(raw_file,    header=True, index=False, sep=',')

In [15]:
sql = '''
SELECT *
FROM exempts
ORDER BY name'''
df_exempts = pd.read_sql(sql, conlt)
df_exempts.shape[0]

403

In [16]:
df_merge = pd.merge(df, df_exempts, on='name', how='outer', indicator=True)
df_merge.shape

(406, 7)

### Tickers that are in Exempts table

In [17]:
in_exempts = df_merge.loc[
    df_merge['_merge'] == 'both',
    ['name','year','quarter','published','link']
    
]
in_exempts.year = in_exempts.year.astype(int)
in_exempts.quarter = in_exempts.quarter.astype(int)
in_exempts.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link


In [18]:
in_exempts.sort_values(by=['published'],ascending=[False]).shape[0]

0

### Not in exempts table

In [19]:
df_out = df_merge.loc[
    df_merge['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]
df_out.year = df_out.year.astype(int)
df_out.quarter = df_out.quarter.astype(int)
df_out.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link
0,COTTO,2022,4,"Tue, 24 Jan 2023 13:38:21 +0700",https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174321600
1,SCGP,2022,4,"Tue, 24 Jan 2023 12:37:52 +0700",https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174320110
2,MASTER,2022,3,"Tue, 24 Jan 2023 08:32:02 +0700",https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174317110


In [20]:
#df_out = df_out.drop(df_out.index[df_out['name'] == "SCC"])
df_out.shape[0]

3

In [21]:
sql = '''
SELECT *
FROM tickers
ORDER BY name'''
df_tickers = pd.read_sql(sql, conlt)
df_tickers.shape

(401, 9)

In [22]:
df_merge2 = pd.merge(df_out, df_tickers, on='name', how='outer', indicator=True)
df_merge2.shape

(402, 14)

### There are no ticker records

In [23]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]

,name,year,quarter,published,link
2,MASTER,2022.0,3.0,"Tue, 24 Jan 2023 08:32:02 +0700",https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174317110


In [24]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link','id','market']
].shape

(1, 7)

### There are ticker records

In [25]:
df_out2 = df_merge2.loc[
    df_merge2['_merge'] == 'both',
    ['name','year','quarter','published','link','id','market']
]
df_out2

,name,year,quarter,published,link,id,market
0,COTTO,2022.0,4.0,"Tue, 24 Jan 2023 13:38:21 +0700",https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174321600,710.0,SET
1,SCGP,2022.0,4.0,"Tue, 24 Jan 2023 12:37:52 +0700",https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174320110,713.0,SET50 / SETCLMV / SETTHSI


In [26]:
df_out2 = df_out2[df_out2.year.notnull()]
df_out2.shape

(2, 7)

In [27]:
df_out2['year'] = df_out2['year'].astype(int)
df_out2['quarter'] = df_out2['quarter'].astype(int)
df_out2.shape

(2, 7)

In [28]:
file_name = 'F45-CLEAN-' + mmdd_str + '.csv'
clean_file = clean_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(output_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(clean_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(box_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(one_file, header=True, index=False, sep=',')

In [29]:
sql = '''
SELECT * 
FROM epss
WHERE year = 2022'''
df_epss = pd.read_sql(sql, conlt)
df_epss.shape

(698, 14)

In [30]:
df_merge3 = pd.merge(df_out2, df_epss, on=['name','year','quarter'], how='outer', indicator=True)
df_merge3.shape

(700, 19)

### Already input, display profit amt & eps to check with new F45

In [31]:
df_merge3[df_merge3['_merge'] == 'both']

,name,year,quarter,published,link,id_x,market,id_y,q_amt,y_amt,aq_amt,ay_amt,q_eps,y_eps,aq_eps,ay_eps,ticker_id,publish_date,_merge


In [32]:
df_merge3[df_merge3['_merge'] == 'both'].shape

(0, 19)

### New F-45

In [33]:
df_inp2 = df_merge3[df_merge3['_merge'] == 'left_only']
df_inp3 = df_inp2.copy()
df_inp3.shape

(2, 19)

In [34]:
df_inp3['year'] = df_inp3['year'].astype(str)
df_inp3['quarter'] = df_inp3['quarter'].astype(str)
final = df_inp3.sort_values('name',ascending=True)
final_str = final.name+' '+final.year+' '+final.quarter+' '+final.link
final_str

0    COTTO 2022 4 https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174321600
1     SCGP 2022 4 https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174320110
dtype: object

In [35]:
df_inp3_q3 = df_inp3[df_inp3['quarter'] == '4']
final_q3 = df_inp3_q3.sort_values('name',ascending=True)
final_str_q3 = final_q3.name+' '+final_q3.year+' '+final_q3.quarter+' '+final_q3.link
final_str_q3

0    COTTO 2022 4 https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174321600
1     SCGP 2022 4 https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16745174320110
dtype: object

In [32]:
df_inp3_q1 = df_inp3[df_inp3['quarter'] != '4']
final_q1 = df_inp3_q1.sort_values('name',ascending=True)
final_str_q1 = final_q1.name+' '+final_q1.year+' '+final_q1.quarter+' '+final_q1.link
final_str_q1

3    TSTH 2022 3 https://classic.set.or.th/set/newsdetails.do?language=en&country=US&newsId=16740853998220
dtype: object